In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from catboost import CatBoostRegressor

# Load dataset
df = pd.read_csv("../dataset/CarWise_cleaned.csv")

print("Dataset shape:", df.shape)
print("Target column: price")
print("Missing values:", df.isnull().sum().sum())

df.head()

Dataset shape: (8249, 528)
Target column: price
Missing values: 0


,transmission,ownerNo,modelYear,price,Kms Driven,Mileage,Seats,Fuel type_Diesel,Fuel type_Electric,Fuel type_Lpg,...,Color_g brown,Color_golden brown,Color_m grey,Color_magma gray,Color_urban titanim,City_Chennai,City_Delhi,City_Hyderabad,City_Jaipur,City_Kolkata
0,1,2.577392,-0.391413,400000.0,0.819543,1.001444,-0.305294,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1.008900,0.379360,811000.0,-0.352937,-0.587173,-0.305294,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,-0.559592,0.379360,585000.0,-0.631733,1.194161,-0.305294,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,-0.559592,-0.648338,462000.0,-0.553226,-0.040272,-0.305294,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,-0.559592,-0.391413,790000.0,0.013659,1.144680,-0.305294,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [11]:
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop(columns=["price"]).copy()
y = df["price"].copy()

# Identify numerical columns that need scaling
numeric_cols = [
    "ownerNo",
    "modelYear",
    "Kms Driven",
    "Mileage",
    "Seats"
]

# Fit a new scaler on the existing feature values
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

print("Scaling completed!")

Scaling completed!


In [2]:
X = df.drop(columns=["price"])
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Number of features:", X_train.shape[1])

Training rows: 6599
Testing rows: 1650
Number of features: 527


In [3]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ),

    "CatBoost": CatBoostRegressor(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="RMSE",
        verbose=False,
        random_seed=42,
        thread_count=-1
    )
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

    trained_models[name] = model

    print(f"{name} completed!")

Training Linear Regression...
Linear Regression completed!
Training Random Forest...
Random Forest completed!
Training CatBoost...
CatBoost completed!


In [4]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="MAE")

display(results_df)

,Model,MAE,RMSE,R2 Score
1,Random Forest,170550.501628,660116.144217,0.851286
2,CatBoost,198139.905097,610121.773501,0.872959
0,Linear Regression,261069.507254,724250.302331,0.820986


In [5]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/best_price_model.joblib")

print("Best Model:", best_model_name)
print("Model saved successfully!")

Best Model: Random Forest
Model saved successfully!


In [6]:
predictions = best_model.predict(X_test)

comparison = pd.DataFrame({
    "Actual Price": y_test.values[:10],
    "Predicted Price": predictions[:10]
})

comparison["Difference"] = abs(
    comparison["Actual Price"] - comparison["Predicted Price"]
)

display(comparison.round(2))

,Actual Price,Predicted Price,Difference
0,345000.0,288126.67,56873.33
1,150000.0,136671.49,13328.51
2,140000.0,171446.67,31446.67
3,415000.0,406526.67,8473.33
4,210000.0,253920.00,43920.00
5,900000.0,1133686.67,233686.67
6,425000.0,439553.33,14553.33
7,200000.0,397373.33,197373.33
8,671000.0,654113.33,16886.67
9,674000.0,516453.33,157546.67


In [7]:
import os

print(os.path.exists("../models/best_price_model.joblib"))

True


In [8]:
print("Total features:", len(X.columns))

print("\nFirst 40 feature names:")
print(X.columns[:40].tolist())

print("\nLast 20 feature names:")
print(X.columns[-20:].tolist())

print("\nNumeric feature ranges:")
display(X.describe().loc[["min", "max"]].T.head(15))

Total features: 527

First 40 feature names:
['transmission', 'ownerNo', 'modelYear', 'Kms Driven', 'Mileage', 'Seats', 'Fuel type_Diesel', 'Fuel type_Electric', 'Fuel type_Lpg', 'Fuel type_Petrol', 'body type_Coupe', 'body type_Hatchback', 'body type_Hybrids', 'body type_MUV', 'body type_Minivans', 'body type_Pickup Trucks', 'body type_SUV', 'body type_Sedan', 'body type_Wagon', 'Brand_BMW', 'Brand_Chevrolet', 'Brand_Citroen', 'Brand_Datsun', 'Brand_Fiat', 'Brand_Ford', 'Brand_Hindustan Motors', 'Brand_Honda', 'Brand_Hyundai', 'Brand_Isuzu', 'Brand_Jaguar', 'Brand_Jeep', 'Brand_Kia', 'Brand_Land Rover', 'Brand_Lexus', 'Brand_MG', 'Brand_Mahindra', 'Brand_Mahindra Renault', 'Brand_Mahindra Ssangyong', 'Brand_Maruti', 'Brand_Mercedes-Benz']

Last 20 feature names:
['Color_White', 'Color_Wine Red', 'Color_Yellow', 'Color_a silver', 'Color_b grey', 'Color_b red', 'Color_beige', 'Color_c bronze', 'Color_c brown', 'Color_chill', 'Color_g brown', 'Color_golden brown', 'Color_m grey', 'Color_

,min,max
transmission,0.000000,1.000000
ownerNo,-2.128084,5.714376
modelYear,-8.099143,1.663981
Kms Driven,-0.790868,73.080462
Mileage,-3.170629,31.445599
Seats,-4.834733,7.243771
Fuel type_Diesel,0.000000,1.000000
Fuel type_Electric,0.000000,1.000000
Fuel type_Lpg,0.000000,1.000000
Fuel type_Petrol,0.000000,1.000000


In [9]:
numeric_cols = [
    "transmission",
    "ownerNo",
    "modelYear",
    "Kms Driven",
    "Mileage",
    "Seats"
]

display(df[numeric_cols].describe().T)

print("Unique transmission values:")
print(df["transmission"].unique())

print("\nUnique owner values:")
print(df["ownerNo"].unique())

,count,mean,std,min,25%,50%,75%,max
transmission,8249.0,7.236029e-01,0.447243,0.000000,0.000000,1.000000,1.000000,1.000000
ownerNo,8249.0,-4.651389e-17,1.000061,-2.128084,-0.559592,-0.559592,1.008900,5.714376
modelYear,8249.0,-8.972013e-15,1.000061,-8.099143,-0.648338,0.122435,0.636284,1.663981
Kms Driven,8249.0,-1.205916e-17,1.000061,-0.790868,-0.389283,-0.070273,0.282287,73.080462
Mileage,8249.0,-3.100926e-16,1.000061,-3.170629,-0.584569,-0.058502,0.558715,31.445599
Seats,8249.0,-6.051113e-16,1.000061,-4.834733,-0.305294,-0.305294,-0.305294,7.243771


Unique transmission values:
[1 0]

Unique owner values:
[ 2.57739223  1.00890027 -0.55959169  4.14588419  5.71437615 -2.12808365]


In [10]:
categories = ["Fuel type_", "body type_", "Brand_", "model_",
              "Insurance Type_", "Color_", "City_"]

for prefix in categories:
    values = [col.replace(prefix, "") for col in X.columns
              if col.startswith(prefix)]
    print(f"\n{prefix}")
    print(values)


Fuel type_
['Diesel', 'Electric', 'Lpg', 'Petrol']

body type_
['Coupe', 'Hatchback', 'Hybrids', 'MUV', 'Minivans', 'Pickup Trucks', 'SUV', 'Sedan', 'Wagon']

Brand_
['BMW', 'Chevrolet', 'Citroen', 'Datsun', 'Fiat', 'Ford', 'Hindustan Motors', 'Honda', 'Hyundai', 'Isuzu', 'Jaguar', 'Jeep', 'Kia', 'Land Rover', 'Lexus', 'MG', 'Mahindra', 'Mahindra Renault', 'Mahindra Ssangyong', 'Maruti', 'Mercedes-Benz', 'Mini', 'Mitsubishi', 'Nissan', 'Opel', 'Porsche', 'Renault', 'Skoda', 'Tata', 'Toyota', 'Volkswagen', 'Volvo']

model_
['Audi A3', 'Audi A3 cabriolet', 'Audi A4', 'Audi A6', 'Audi A8', 'Audi Q2', 'Audi Q3', 'Audi Q3 Sportback', 'Audi Q5', 'Audi Q7', 'Audi S5 Sportback', 'BMW 1 Series', 'BMW 2 Series', 'BMW 3 Series', 'BMW 3 Series GT', 'BMW 3 Series Gran Limousine', 'BMW 5 Series', 'BMW 6 Series', 'BMW 7 Series', 'BMW X1', 'BMW X3', 'BMW X4', 'BMW X5', 'BMW X7', 'Chevrolet Aveo', 'Chevrolet Aveo U-VA', 'Chevrolet Beat', 'Chevrolet Captiva', 'Chevrolet Cruze', 'Chevrolet Enjoy', 'Chev

In [12]:
import pandas as pd
import numpy as np

# Load the new raw dataset
df = pd.read_csv("../dataset/CarDekho_raw.csv")

print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nFirst 5 Records:")
display(df.head())

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Records:", df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

Dataset Shape: (8128, 13)

Column Names:
['name', 'year', 'selling_price', 'km_driven', 'fuel', 'seller_type', 'transmission', 'owner', 'mileage', 'engine', 'max_power', 'torque', 'seats']

First 5 Records:


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0



Missing Values:


name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          221
engine           221
max_power        215
torque           222
seats            221
dtype: int64


Duplicate Records: 1202

Data Types:
name                 str
year               int64
selling_price      int64
km_driven          int64
fuel                 str
seller_type          str
transmission         str
owner                str
mileage              str
engine               str
max_power            str
torque               str
seats            float64
dtype: object


In [13]:
print("Price statistics:")
display(df["selling_price"].describe())

print("\nUnique fuel types:")
print(df["fuel"].unique())

print("\nUnique transmissions:")
print(df["transmission"].unique())

print("\nUnique owners:")
print(df["owner"].unique())

Price statistics:


count    8.128000e+03
mean     6.382718e+05
std      8.062534e+05
min      2.999900e+04
25%      2.549990e+05
50%      4.500000e+05
75%      6.750000e+05
max      1.000000e+07
Name: selling_price, dtype: float64


Unique fuel types:
<StringArray>
['Diesel', 'Petrol', 'LPG', 'CNG']
Length: 4, dtype: str

Unique transmissions:
<StringArray>
['Manual', 'Automatic']
Length: 2, dtype: str

Unique owners:
<StringArray>
[         'First Owner',         'Second Owner',          'Third Owner',
 'Fourth & Above Owner',       'Test Drive Car']
Length: 5, dtype: str


In [14]:
import re
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/CarDekho_raw.csv")

# Remove duplicate records
df = df.drop_duplicates().copy()

# Extract numeric values from columns containing units
numeric_text_cols = ["mileage", "engine", "max_power", "seats"]

for col in numeric_text_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
        errors="coerce"
    )

# Remove torque because its formats are inconsistent
df = df.drop(columns=["torque"])

print("Cleaned dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

display(df.head())

Cleaned dataset shape: (6926, 12)

Missing values:
name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
mileage          208
engine           208
max_power        206
seats            208
dtype: int64


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.00,1396.0,90.00,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.10,1298.0,88.20,5.0


In [15]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["selling_price"])
y = df["selling_price"]

numeric_features = [
    "year", "km_driven", "mileage",
    "engine", "max_power", "seats"
]

categorical_features = [
    "name", "fuel", "seller_type",
    "transmission", "owner"
]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Features:", X.shape[1])

Training records: 5540
Testing records: 1386
Features: 11


In [16]:
import os
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Numerical preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Complete model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=2
    ))
])

print("Pipeline created successfully!")

Pipeline created successfully!


In [17]:
print("Training Random Forest...")

model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print("\nMODEL EVALUATION")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

Training Random Forest...

MODEL EVALUATION
MAE: 73035.94
RMSE: 127072.84
R2 Score: 0.9264


In [18]:
os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/car_price_pipeline.joblib")

print("Complete pipeline saved successfully!")

Complete pipeline saved successfully!


In [19]:

import sys
sys.path.append("..")

from src.price_prediction.predict_price import predict_car_price

sample_car = {
    "name": "Maruti Swift Dzire VDI",
    "year": 2018,
    "km_driven": 45000,
    "fuel": "Diesel",
    "seller_type": "Individual",
    "transmission": "Manual",
    "owner": "First Owner",
    "mileage": 23.4,
    "engine": 1248,
    "max_power": 74,
    "seats": 5
}

price = predict_car_price(sample_car)

print("Predicted Car Price: ₹", price)


Predicted Car Price: ₹ 669136.66


In [20]:
import pandas as pd
import numpy as np

# Load the new dataset
new_df = pd.read_csv("../dataset/used_cars_dataset_v2.csv")

print("DATASET SHAPE:", new_df.shape)

print("\nCOLUMN NAMES:")
print(new_df.columns.tolist())

print("\nFIRST 5 RECORDS:")
display(new_df.head())

print("\nDATA TYPES:")
print(new_df.dtypes)

print("\nMISSING VALUES:")
display(new_df.isnull().sum())

print("\nDUPLICATE RECORDS:", new_df.duplicated().sum())

print("\nNUMERICAL STATISTICS:")
display(new_df.describe().T)

DATASET SHAPE: (14993, 11)

COLUMN NAMES:
['Brand', 'model', 'Year', 'Age', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'PostedDate', 'AdditionInfo', 'AskPrice']

FIRST 5 RECORDS:


,Brand,model,Year,Age,kmDriven,Transmission,Owner,FuelType,PostedDate,AdditionInfo,AskPrice
0,Honda,City,2001,23,"98,000 km",Manual,second,Petrol,Nov-24,"Honda City v teck in mint condition, valid gen...","₹ 1,95,000"
1,Toyota,Innova,2009,15,190000.0 km,Manual,second,Diesel,Jul-24,"Toyota Innova 2.5 G (Diesel) 7 Seater, 2009, D...","₹ 3,75,000"
2,Volkswagen,VentoTest,2010,14,"77,246 km",Manual,first,Diesel,Nov-24,"Volkswagen Vento 2010-2013 Diesel Breeze, 2010...","₹ 1,84,999"
3,Maruti Suzuki,Swift,2017,7,"83,500 km",Manual,second,Diesel,Nov-24,Maruti Suzuki Swift 2017 Diesel Good Condition,"₹ 5,65,000"
4,Maruti Suzuki,Baleno,2019,5,"45,000 km",Automatic,first,Petrol,Nov-24,"Maruti Suzuki Baleno Alpha CVT, 2019, Petrol","₹ 6,85,000"



DATA TYPES:
Brand             str
model             str
Year            int64
Age             int64
kmDriven          str
Transmission      str
Owner             str
FuelType          str
PostedDate        str
AdditionInfo      str
AskPrice          str
dtype: object

MISSING VALUES:


Brand            0
model            0
Year             0
Age              0
kmDriven        88
Transmission     0
Owner            0
FuelType         0
PostedDate       0
AdditionInfo     0
AskPrice         0
dtype: int64


DUPLICATE RECORDS: 997

NUMERICAL STATISTICS:


,count,mean,std,min,25%,50%,75%,max
Year,14993.0,2016.32215,4.343017,1900.0,2014.0,2017.0,2019.0,2024.0
Age,14993.0,7.67785,4.343017,0.0,5.0,7.0,10.0,124.0


In [21]:
# Inspect suspicious manufacturing years
print("Years below 1990:")
display(df[df["Year"] < 1990][
    ["Brand", "model", "Year", "Age", "AskPrice"]
].head(20))

# Inspect suspicious vehicle ages
print("\nVehicles older than 35 years:")
display(df[df["Age"] > 35][
    ["Brand", "model", "Year", "Age", "AskPrice"]
].head(20))

# Inspect asking price statistics
print("\nAsking price statistics:")
display(df["AskPrice"].describe())

print("\nPrice percentiles:")
display(df["AskPrice"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

# Inspect unusually high prices
print("\nTop 15 asking prices:")
display(df.nlargest(15, "AskPrice")[
    ["Brand", "model", "Year", "Age", "kmDriven", "AskPrice"]
])

Years below 1990:


KeyError: 'Year'

In [22]:
# Reload original dataset and repeat essential cleaning
df = pd.read_csv("../dataset/used_cars_dataset_v2.csv")

df["AskPrice"] = pd.to_numeric(
    df["AskPrice"].astype(str)
    .str.replace(r"[₹,\s]", "", regex=True),
    errors="coerce"
)

df["kmDriven"] = pd.to_numeric(
    df["kmDriven"].astype(str)
    .str.replace(r"[^\d.]", "", regex=True),
    errors="coerce"
)

df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

df = df.drop_duplicates().copy()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (13985, 11)
Columns: ['Brand', 'model', 'Year', 'Age', 'kmDriven', 'Transmission', 'Owner', 'FuelType', 'PostedDate', 'AdditionInfo', 'AskPrice']


In [23]:
print("Years below 1990:")

display(df.loc[df["Year"] < 1990, [
    "Brand", "model", "Year", "Age", "AskPrice"
]].head(20))

print("\nVehicles older than 35 years:")

display(df.loc[df["Age"] > 35, [
    "Brand", "model", "Year", "Age", "AskPrice"
]].head(20))

print("\nPrice statistics:")
display(df["AskPrice"].describe())

print("\nPrice percentiles:")
display(df["AskPrice"].quantile(
    [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
))

print("\nTop 15 asking prices:")
display(df.nlargest(15, "AskPrice")[[
    "Brand", "model", "Year", "Age", "kmDriven", "AskPrice"
]])

Years below 1990:


,Brand,model,Year,Age,AskPrice
7404,Maruti Suzuki,800,1986,38,120000
11260,Maruti Suzuki,Wagon-R,1931,93,349000
11392,Renault,Triber,1900,124,440000
11458,Mahindra,Jeep,1978,46,200000
12545,Maruti Suzuki,Swift,1920,104,519000



Vehicles older than 35 years:


,Brand,model,Year,Age,AskPrice
7404,Maruti Suzuki,800,1986,38,120000
11260,Maruti Suzuki,Wagon-R,1931,93,349000
11392,Renault,Triber,1900,124,440000
11458,Mahindra,Jeep,1978,46,200000
12545,Maruti Suzuki,Swift,1920,104,519000



Price statistics:


count    1.398500e+04
mean     9.797811e+05
std      1.586030e+06
min      1.500000e+04
25%      3.400000e+05
50%      5.600000e+05
75%      9.900000e+05
max      4.250000e+07
Name: AskPrice, dtype: float64


Price percentiles:


0.01      79839.16
0.05     148000.00
0.25     340000.00
0.50     560000.00
0.75     990000.00
0.95    3200000.00
0.99    7495640.00
Name: AskPrice, dtype: float64


Top 15 asking prices:


,Brand,model,Year,Age,kmDriven,AskPrice
3331,Rolls-Royce,Phantom Series II,2015,9,11500.0,42500000
12685,Lamborghini,Urus,2021,3,26000.0,39000000
13286,Mercedes-Benz,G Class,2023,1,6000.0,34800000
10291,Mercedes-Benz,G Class,2024,0,3500.0,30500000
962,Lexus,LX,2024,0,24000.0,27750000
331,Aston Martin,Vanquish,2016,8,25700.0,26400000
811,Mercedes-Benz,G Class,2019,5,59000.0,26000000
5767,Mercedes-Benz,G Class,2019,5,59000.0,24900000
1187,Toyota,Land Cruiser,2023,1,15000.0,24700000
7435,Mercedes-Benz,SL-Class,2024,0,1400.0,24500000


In [24]:
# Remove impossible manufacturing years
df = df[
    (df["Year"] >= 1990) &
    (df["Year"] <= 2024)
].copy()

# Convert listing date into a date
df["PostedDate"] = pd.to_datetime(
    df["PostedDate"],
    format="%b-%y",
    errors="coerce"
)

# Calculate vehicle age at the time of listing
df["CarAge"] = df["PostedDate"].dt.year - df["Year"]

# Remove invalid ages
df = df[
    (df["CarAge"] >= 0) &
    (df["CarAge"] <= 35)
].copy()

# Remove invalid asking prices
df = df[
    df["AskPrice"].notna() &
    (df["AskPrice"] >= 20000) &
    (df["AskPrice"] <= 100000000)
].copy()

# Remove invalid kilometre readings
df.loc[
    (df["kmDriven"] < 0) | (df["kmDriven"] > 1000000),
    "kmDriven"
] = np.nan

# Remove records with missing essential information
df = df.dropna(subset=[
    "Brand", "model", "Year", "AskPrice",
    "Transmission", "Owner", "FuelType"
])

# Remove unnecessary columns
df = df.drop(columns=["Age", "PostedDate", "AdditionInfo"])

print("Final cleaned dataset:", df.shape)
print("\nMissing values:")
display(df.isnull().sum())

print("\nPrice distribution:")
display(df["AskPrice"].describe())

Final cleaned dataset: (13974, 9)

Missing values:


Brand            0
model            0
Year             0
kmDriven        86
Transmission     0
Owner            0
FuelType         0
AskPrice         0
CarAge           0
dtype: int64


Price distribution:


count    1.397400e+04
mean     9.804289e+05
std      1.586478e+06
min      2.000000e+04
25%      3.400000e+05
50%      5.600000e+05
75%      9.900000e+05
max      4.250000e+07
Name: AskPrice, dtype: float64

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Features and target
features = [
    "Brand",
    "model",
    "Year",
    "CarAge",
    "kmDriven",
    "Transmission",
    "Owner",
    "FuelType"
]

X = df[features]
y = df["AskPrice"]

numeric_features = ["Year", "CarAge", "kmDriven"]

categorical_features = [
    "Brand", "model", "Transmission", "Owner", "FuelType"
]

# Split before fitting preprocessing to avoid test-data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )
}

results = []
trained_models = {}

for name, regressor in models.items():

    print("Training:", name)

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", regressor)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("MAE")

display(results_df.round(2))

Training: Random Forest
Training: Extra Trees


,Model,MAE,RMSE,R2
1,Extra Trees,204116.11,527188.29,0.86
0,Random Forest,215350.16,543694.66,0.85


In [26]:
from catboost import CatBoostRegressor

# Prepare CatBoost input
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

cat_cols = [
    "Brand", "model", "Transmission", "Owner", "FuelType"
]

# CatBoost requires categorical values as strings
for col in cat_cols:
    X_train_cb[col] = X_train_cb[col].fillna("Unknown").astype(str)
    X_test_cb[col] = X_test_cb[col].fillna("Unknown").astype(str)

# Train CatBoost
cat_model = CatBoostRegressor(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    verbose=100,
    random_seed=42,
    thread_count=-1
)

cat_model.fit(
    X_train_cb,
    y_train,
    cat_features=cat_cols
)

# Evaluate
cat_predictions = cat_model.predict(X_test_cb)

cat_mae = mean_absolute_error(y_test, cat_predictions)
cat_rmse = np.sqrt(mean_squared_error(y_test, cat_predictions))
cat_r2 = r2_score(y_test, cat_predictions)

results.append({
    "Model": "CatBoost",
    "MAE": cat_mae,
    "RMSE": cat_rmse,
    "R2": cat_r2
})

results_df = pd.DataFrame(results).sort_values("MAE")

display(results_df.round(2))

0:	learn: 1593783.7015480	total: 60.2ms	remaining: 48.1s
100:	learn: 862163.9680206	total: 5.77s	remaining: 40s
200:	learn: 769006.2732195	total: 16.4s	remaining: 48.9s
300:	learn: 703109.3370584	total: 25.5s	remaining: 42.3s
400:	learn: 648398.5223728	total: 34.4s	remaining: 34.2s
500:	learn: 603805.0088976	total: 40.4s	remaining: 24.1s
600:	learn: 567869.7978030	total: 46.2s	remaining: 15.3s
700:	learn: 528782.6375670	total: 52.1s	remaining: 7.36s
799:	learn: 499550.9545928	total: 58s	remaining: 0us


,Model,MAE,RMSE,R2
1,Extra Trees,204116.11,527188.29,0.86
0,Random Forest,215350.16,543694.66,0.85
2,CatBoost,275330.62,879440.61,0.61


In [27]:
# Use Extra Trees predictions for this initial analysis
best_predictions = trained_models["Extra Trees"].predict(X_test)

evaluation = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": best_predictions
})

evaluation["Absolute Error"] = abs(
    evaluation["Actual Price"] - evaluation["Predicted Price"]
)

evaluation["Price Segment"] = pd.cut(
    evaluation["Actual Price"],
    bins=[0, 300000, 600000, 1000000, 3000000, float("inf")],
    labels=["Below 3L", "3L-6L", "6L-10L", "10L-30L", "Above 30L"]
)

segment_results = evaluation.groupby(
    "Price Segment", observed=False
).agg(
    Cars=("Actual Price", "count"),
    MAE=("Absolute Error", "mean")
)

display(segment_results.round(2))

,Cars,MAE
Price Segment,,
Below 3L,563,95360.67
3L-6L,913,72722.35
6L-10L,624,171803.30
10L-30L,531,352126.66
Above 30L,164,952661.31


In [28]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.base import clone

# Create a fresh Extra Trees pipeline
base_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("regressor", ExtraTreesRegressor(
        n_estimators=300,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

# Apply log transformation to target price
log_model = TransformedTargetRegressor(
    regressor=base_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("Training log-transformed Extra Trees...")

log_model.fit(X_train, y_train)

log_predictions = log_model.predict(X_test)

log_mae = mean_absolute_error(y_test, log_predictions)
log_rmse = np.sqrt(mean_squared_error(y_test, log_predictions))
log_r2 = r2_score(y_test, log_predictions)

print("\nLOG EXTRA TREES RESULTS")
print("MAE:", round(log_mae, 2))
print("RMSE:", round(log_rmse, 2))
print("R2:", round(log_r2, 4))

Training log-transformed Extra Trees...

LOG EXTRA TREES RESULTS
MAE: 207955.68
RMSE: 567891.6
R2: 0.8364


In [29]:
comparison = pd.DataFrame([
    {
        "Model": "Original Extra Trees",
        "MAE": mean_absolute_error(
            y_test,
            trained_models["Extra Trees"].predict(X_test)
        ),
        "RMSE": np.sqrt(mean_squared_error(
            y_test,
            trained_models["Extra Trees"].predict(X_test)
        )),
        "R2": r2_score(
            y_test,
            trained_models["Extra Trees"].predict(X_test)
        )
    },
    {
        "Model": "Log Extra Trees",
        "MAE": log_mae,
        "RMSE": log_rmse,
        "R2": log_r2
    }
])

display(comparison.round(2))

,Model,MAE,RMSE,R2
0,Original Extra Trees,204116.11,527188.29,0.86
1,Log Extra Trees,207955.68,567891.60,0.84


In [30]:
import joblib
import os

# Select Random Forest pipeline
final_model = trained_models["Random Forest"]

# Save model
os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_model,
    "../models/carwise_random_forest.joblib"
)

print("Random Forest model saved successfully!")

Random Forest model saved successfully!
